<a href="https://colab.research.google.com/github/SANDHIYA1604/Safeguard-AI/blob/main/SafeGuard_AI_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

def generate_sample(target_risk):
    if target_risk == "LOW":
        temperature = np.random.normal(26, 3)
        gas_level = np.random.normal(200, 50)
        movement = np.random.choice(["Medium", "High"], p=[0.5, 0.5])
        immobility_duration = np.random.exponential(10)
        accel_mag_offset = np.random.normal(0, 0.3)

    elif target_risk == "MEDIUM":
        temperature = np.random.normal(32, 3)
        gas_level = np.random.normal(350, 50)
        movement = np.random.choice(["Low", "Medium"], p=[0.4, 0.6])
        immobility_duration = np.random.exponential(25)
        accel_mag_offset = np.random.normal(0, 0.6)

    elif target_risk == "HIGH":
        temperature = np.random.normal(37, 2)
        gas_level = np.random.normal(480, 40)
        movement = np.random.choice(["Low", "Medium"], p=[0.7, 0.3])
        immobility_duration = np.random.exponential(45)
        accel_mag_offset = np.random.normal(0, 1.5)

    else:  # CRITICAL
        temperature = np.random.normal(41, 2)
        gas_level = np.random.normal(600, 40)
        movement = "Low"
        immobility_duration = np.random.exponential(75)
        accel_mag_offset = np.random.choice([np.random.normal(0, 3), np.random.normal(-5, 1)])

    accel_x = np.random.normal(0, 0.5)
    accel_y = np.random.normal(0, 0.5)
    accel_z = 9.8 + accel_mag_offset

    return {
        "temperature": round(temperature, 2),
        "gas_level": round(max(gas_level, 0), 2),
        "accel_x": round(accel_x, 2),
        "accel_y": round(accel_y, 2),
        "accel_z": round(accel_z, 2),
        "movement": movement,
        "immobility_duration": round(max(immobility_duration, 0), 2),
        "risk_level": target_risk
    }

# Generate equal numbers per class
n_per_class = 1250
labels = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
data = []
for label in labels:
    data.extend([generate_sample(label) for _ in range(n_per_class)])

df = pd.DataFrame(data).sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle rows

df.to_csv("safeguard_ai_dataset.csv", index=False)
print(df["risk_level"].value_counts())
df.head(10)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

# 1. Encode categorical column (movement)
movement_map = {"Low": 0, "Medium": 1, "High": 2}
df["movement_encoded"] = df["movement"].map(movement_map)

# 2. Encode target labels
label_encoder = LabelEncoder()
df["risk_encoded"] = label_encoder.fit_transform(df["risk_level"])
# This maps: CRITICAL=0, HIGH=1, LOW=2, MEDIUM=3 (alphabetical order — we'll check this)
print("Label mapping:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

# 3. Select features
feature_cols = ["temperature", "gas_level", "accel_x", "accel_y", "accel_z", "movement_encoded", "immobility_duration"]
X = df[feature_cols].values
y = df["risk_encoded"].values

# 4. Scale features (DNNs train better on normalized data)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 5. Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# 6. Build the DNN
model = tf.keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(4, activation="softmax")  # 4 output classes
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

model.summary()

# 7. Train
history = model.fit(X_train, y_train, epochs=30, batch_size=32, validation_split=0.2, verbose=1)

# 8. Evaluate
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Accuracy: {test_acc*100:.2f}%")

In [ ]:
import gradio as gr
import numpy as np

risk_labels = {0: "CRITICAL", 1: "HIGH", 2: "LOW", 3: "MEDIUM"}
risk_grad = {
    0: ("#eb3b5a", "#c0392b"),
    1: ("#f7931e", "#e67e22"),
    2: ("#20bf6b", "#0f9b5f"),
    3: ("#f6c744", "#e0a800"),
}
risk_emojis = {0: "🔴", 1: "🟠", 2: "🟢", 3: "🟡"}

last_result = {"risk": "—", "confidence": 0, "temp": "—", "gas": "—"}

def predict_risk(temperature, gas_level, accel_x, accel_y, accel_z, movement, immobility_duration):
    movement_encoded = movement_map[movement]
    features = np.array([[temperature, gas_level, accel_x, accel_y, accel_z, movement_encoded, immobility_duration]])
    features_scaled = scaler.transform(features)

    prediction = model.predict(features_scaled, verbose=0)
    predicted_class = np.argmax(prediction)
    confidence = prediction[0][predicted_class] * 100

    risk = risk_labels[predicted_class]
    g1, g2 = risk_grad[predicted_class]
    emoji = risk_emojis[predicted_class]
    explanation = generate_explanation(risk, temperature, gas_level, movement, immobility_duration)
    conf_str = "{:.1f}".format(confidence)

    last_result["risk"] = risk
    last_result["confidence"] = conf_str
    last_result["temp"] = temperature
    last_result["gas"] = gas_level

    html = (
        '<div style="font-family:Segoe UI, sans-serif;">'
        '<div style="display:grid; grid-template-columns:repeat(4,1fr); gap:14px; margin-bottom:16px;">'
            '<div style="background:linear-gradient(135deg,#6a5acd,#836fff); border-radius:14px; padding:18px; color:white;">'
            '<div style="font-size:12px; opacity:0.85;">Temperature</div>'
            '<div style="font-size:26px; font-weight:800; margin-top:6px;">' + str(temperature) + '&deg;C</div></div>'
            '<div style="background:linear-gradient(135deg,#2196f3,#21c4f3); border-radius:14px; padding:18px; color:white;">'
            '<div style="font-size:12px; opacity:0.85;">Gas Level</div>'
            '<div style="font-size:26px; font-weight:800; margin-top:6px;">' + str(gas_level) + '</div></div>'
            '<div style="background:linear-gradient(135deg,' + g1 + ',' + g2 + '); border-radius:14px; padding:18px; color:white;">'
            '<div style="font-size:12px; opacity:0.85;">Risk Level</div>'
            '<div style="font-size:22px; font-weight:800; margin-top:6px;">' + emoji + ' ' + risk + '</div></div>'
            '<div style="background:linear-gradient(135deg,#eb3b5a,#c0392b); border-radius:14px; padding:18px; color:white;">'
            '<div style="font-size:12px; opacity:0.85;">Immobility</div>'
            '<div style="font-size:26px; font-weight:800; margin-top:6px;">' + str(immobility_duration) + 's</div></div>'
        '</div>'
        '<div style="background:linear-gradient(135deg,#1a1a2e,#16213e); border-radius:14px; padding:22px; color:white; margin-bottom:16px;">'
        '<div style="font-size:13px; opacity:0.7; margin-bottom:4px;">Model Confidence</div>'
        '<div style="font-size:34px; font-weight:800; color:#f6c744;">' + conf_str + '%</div>'
        '<div style="height:8px; background:#ffffff22; border-radius:4px; overflow:hidden; margin-top:10px;">'
        '<div style="height:100%; width:' + conf_str + '%; background:linear-gradient(90deg,' + g1 + ',' + g2 + ');"></div></div></div>'
        '<div style="background:white; border:1px solid #e5e5e5; border-radius:14px; padding:20px;">'
        '<div style="font-weight:700; font-size:14px; color:#6a5acd; margin-bottom:8px;">📋 AI Explanation</div>'
        '<div style="font-size:14px; line-height:1.6; color:#333;">' + explanation + '</div></div>'
        '</div>'
    )
    return html

def get_dashboard_summary():
    r = last_result
    color_map = {"CRITICAL": "#eb3b5a", "HIGH": "#f7931e", "MEDIUM": "#f6c744", "LOW": "#20bf6b", "—": "#999"}
    c = color_map.get(r["risk"], "#999")
    html = (
        '<div style="font-family:Segoe UI, sans-serif;">'
        '<div style="display:grid; grid-template-columns:repeat(3,1fr); gap:14px;">'
            '<div style="background:linear-gradient(135deg,#6a5acd,#836fff); border-radius:14px; padding:20px; color:white;">'
            '<div style="font-size:12px; opacity:0.85;">Last Reading Temp</div>'
            '<div style="font-size:28px; font-weight:800; margin-top:6px;">' + str(r["temp"]) + '</div></div>'
            '<div style="background:linear-gradient(135deg,#2196f3,#21c4f3); border-radius:14px; padding:20px; color:white;">'
            '<div style="font-size:12px; opacity:0.85;">Last Reading Gas</div>'
            '<div style="font-size:28px; font-weight:800; margin-top:6px;">' + str(r["gas"]) + '</div></div>'
            '<div style="background:' + c + '; border-radius:14px; padding:20px; color:white;">'
            '<div style="font-size:12px; opacity:0.85;">Current Risk</div>'
            '<div style="font-size:22px; font-weight:800; margin-top:6px;">' + str(r["risk"]) + '</div></div>'
        '</div></div>'
    )
    return html

custom_css = """
.gradio-container {background:#f4f5fb !important;}
#sidebar {background:linear-gradient(180deg,#6a5acd,#836fff); border-radius:16px; padding:16px; min-height:500px;}
#sidebar button {background:#ffffff22 !important; color:white !important; border:none !important; text-align:left !important; margin-bottom:8px !important;}
#sidebar h2 {color:white; font-size:20px; margin:4px 0 20px 4px;}
#title-block {padding:16px 24px; background:white; border-radius:14px; margin-bottom:16px; border:1px solid #eee;}
#title-block h1 {font-size:24px; margin:0; color:#2d3142; font-weight:800;}
#title-block p {color:#8a8f98; margin-top:2px; font-size:13px;}
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Default(primary_hue="purple")) as demo:
    with gr.Row():
        with gr.Column(scale=1, elem_id="sidebar"):
            gr.HTML("<h2>🛡️ SafeGuard AI</h2>")
            btn_dashboard = gr.Button("🏠 Dashboard")
            btn_monitor = gr.Button("📡 Live Monitor")
            btn_about = gr.Button("ℹ️ About")

        with gr.Column(scale=4):
            gr.HTML("""
            <div id="title-block">
                <h1>Worker Risk Monitoring</h1>
                <p>AIoT-based intelligent risk prediction & emergency support system</p>
            </div>
            """)

            with gr.Group(visible=True) as dashboard_section:
                gr.Markdown("### Dashboard Overview")
                dashboard_html = gr.HTML(get_dashboard_summary())
                refresh_btn = gr.Button("🔄 Refresh Summary")

            with gr.Group(visible=False) as monitor_section:
                gr.Markdown("### Live Monitor")
                with gr.Row():
                    with gr.Column(scale=1):
                        temperature = gr.Slider(15, 45, value=30, label="Temperature (°C)")
                        gas_level = gr.Slider(50, 700, value=300, label="Gas Level")
                        accel_x = gr.Slider(-2, 2, value=0, label="Accel X")
                        accel_y = gr.Slider(-2, 2, value=0, label="Accel Y")
                        accel_z = gr.Slider(5, 14, value=9.8, label="Accel Z")
                        movement = gr.Dropdown(["Low", "Medium", "High"], value="Medium", label="Movement")
                        immobility_duration = gr.Slider(0, 150, value=15, label="Immobility Duration (s)")
                        submit_btn = gr.Button("Analyze Worker Status", variant="primary")
                    with gr.Column(scale=2):
                        output = gr.HTML()

            with gr.Group(visible=False) as about_section:
                gr.Markdown("### About SafeGuard AI")
                gr.HTML("""
                <div style="background:white; border:1px solid #eee; border-radius:14px; padding:20px; font-size:14px; line-height:1.7; color:#333;">
                SafeGuard AI is an AIoT-based intelligent risk prediction and emergency support system for
                government field workers. It combines a Deep Neural Network for real-time risk classification
                with Generative AI (Gemini) to explain each risk and recommend supervisor action.<br><br>
                <b>Pipeline:</b> Sensors &rarr; ESP32 &rarr; DNN &rarr; Gemini &rarr; Dashboard + Alerts
                </div>
                """)

    def show_dashboard():
        return gr.update(visible=True), gr.update(visible=False), gr.update(visible=False)
    def show_monitor():
        return gr.update(visible=False), gr.update(visible=True), gr.update(visible=False)
    def show_about():
        return gr.update(visible=False), gr.update(visible=False), gr.update(visible=True)

    btn_dashboard.click(show_dashboard, outputs=[dashboard_section, monitor_section, about_section])
    btn_monitor.click(show_monitor, outputs=[dashboard_section, monitor_section, about_section])
    btn_about.click(show_about, outputs=[dashboard_section, monitor_section, about_section])

    submit_btn.click(
        fn=predict_risk,
        inputs=[temperature, gas_level, accel_x, accel_y, accel_z, movement, immobility_duration],
        outputs=output
    )
    refresh_btn.click(fn=get_dashboard_summary, outputs=dashboard_html)

demo.launch(share=True)

In [ ]:
!pip install google-generativeai -q

In [ ]:
import requests
import time
from getpass import getpass

api_key = getpass("Enter your Gemini API key: ")

def generate_explanation(risk_level, temperature, gas_level, movement, immobility_duration):
    prompt = f"""A field worker safety monitoring system detected the following:
Risk Level: {risk_level}
Temperature: {temperature}°C
Gas Level: {gas_level}
Movement: {movement}
Immobility Duration: {immobility_duration} seconds

In 2-3 sentences, explain why this risk level was likely triggered and give ONE clear, actionable safety recommendation for the supervisor. Be direct and practical, no preamble."""

    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-3.6-flash:generateContent?key={api_key}"
    payload = {"contents": [{"parts": [{"text": prompt}]}]}

    for attempt in range(3):
        try:
            response = requests.post(url, json=payload, timeout=15)
            response.raise_for_status()
            data = response.json()
            return data["candidates"][0]["content"]["parts"][0]["text"]
        except Exception as e:
            if attempt < 2:
                time.sleep(2)
                continue
            return f"(AI explanation unavailable after retries: {str(e)[:100]})"

# Quick test
explanation = generate_explanation("CRITICAL", 41, 610, "Low", 110)
print(explanation)

In [ ]:
import requests

list_url = f"https://generativelanguage.googleapis.com/v1beta/models?key={api_key}"
response = requests.get(list_url)
models = response.json()

for m in models.get("models", []):
    if "generateContent" in m.get("supportedGenerationMethods", []):
        print(m["name"])